# Predicting Brain Pathology in Alzheimer's Disease using Elastic Net Regression

The notebook details an analysis using elastic net regression to predict brain pathology score (amyloid, tangles, gpath, and NIA-Reagan scores) in Alzheimer's disease based on clinical features (e.g. cognitive test scores, underlying health conditions, demographic) from last visit before deceased. 

The analysis aims to create a baseline model for future comparisons with lstm model.

- Inputs:
    - scaled_train_last.pkl, scaled_test_last.pkl

- Results: 
    - predicit gpath:
      - the selected parameters:
        - alpha: 0.1
        - l1 ratio: 0.3
      - Testing $R^2$: 0.2619
    - predicit tangles:
        - the selected parameters:
          - alpha: 0.1
          - l1 ratio: 0.1
        - Testing $R^2$: 0.3854


# Analysis Steps
- Elastic-Net Regression Used participants' last visit data.
    - Parameter Tuning: GridSearchCV to optimize hyperparameters.
    - Model Fitting: Applied best parameters to fit the elastic net model.
    - Performance Evaluation: Evaluated the model on testing data.

# Set up

In [ ]:
import utils
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV

# Load the pickle files
scaled_train_last = pd.read_pickle("NIAvalueUpdatedVersion_scaled_train_last.pkl")
scaled_test_last = pd.read_pickle("NIAvalueUpdatedVersion_scaled_test_last.pkl")

# 1. Perform GLM elastic-net using last visit

## 1.1 Parameter Tuning with GridSearchCV

In [ ]:
# Define the parameter grid
param_grid = {
    'alpha': np.logspace(-3, 0, 10).astype(float).round(3),
    'l1_ratio': np.arange(0.0, 0.45, 0.05).astype(float).round(2)
}

# Use the custom model wrapper
elastic_net_ridge_switcher = utils.ElasticNetRidgeSwitcher()

# GridSearchCV
grid_search = GridSearchCV(
    estimator=elastic_net_ridge_switcher,
    param_grid=param_grid,
    scoring='r2',
    cv=5,  # Number of cross-validation folds
    n_jobs=-1  # Use all available cores
)

In [ ]:
# List of outcomes to loop through
outcomes = ['amyloid', 'gpath', 'tangles', 'niareagansc']

# Define the predictors and the outcomes
drop_vars = ['projid', 'study', 'fu_year', 'cogdx'] + outcomes
train_predictors = scaled_train_last.drop(columns=drop_vars)

# Save grid search results
grid_search_results = {}

for outcome in ['amyloid', 'gpath', 'tangles', 'niareagansc']:
    print(f"Performing grid search for {outcome}")
    
    # Define the target outcome
    train_outcome = scaled_train_last[outcome]
    
    # Fit the grid search
    grid_search.fit(train_predictors, train_outcome)
    cv_results = pd.DataFrame(grid_search.cv_results_)
    
    # Save results
    grid_search_results[outcome] = {
        "cv_results": cv_results,
        "best_params": grid_search.best_params_,
        "best_score": grid_search.best_score_,
    }

## 1.2 Check GridSearchCV results

In [ ]:
# ===== Change this value to control overall text size =====
FONT_SIZE = 16
# ===== Change this value to control the legend text size separately =====
LEGEND_SIZE = 12

# Apply global font settings
import matplotlib as mpl
mpl.rcParams.update({
    "font.size":        FONT_SIZE,
    "axes.titlesize":   FONT_SIZE,
    "axes.labelsize":   FONT_SIZE,
    "xtick.labelsize":  FONT_SIZE,
    "ytick.labelsize":  FONT_SIZE,
    "font.weight":      "bold",
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
})

# Subplot title mapping
title_map = {
    "amyloid":     "Amyloid-β",
    "tangles":     "Tangles",
    "gpath":       "Global AD Pathology",
    "niareagansc": "Pathologic AD",
}

# Set up the subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

# Order: A=Amyloid-β, B=Tangles, C=Pathologic AD, D=Global AD Pathology
for idx, outcome in enumerate(["amyloid", "tangles", "niareagansc", "gpath"]):
    cv_results = grid_search_results[outcome]["cv_results"]
    ax = axes[idx]

    for l1_ratio in cv_results["param_l1_ratio"].unique():
        subset = cv_results[cv_results["param_l1_ratio"] == l1_ratio]
        ax.plot(subset["param_alpha"], subset["mean_test_score"], label=f"L1 Ratio = {l1_ratio:.2f}")

    ax.set_xscale("log")
    ax.set_xlabel("Alpha (log scale)", fontsize=FONT_SIZE, fontweight="bold")
    ax.set_ylabel("5-fold Cross-validation R²", fontsize=FONT_SIZE, fontweight="bold")
    ax.set_title(title_map[outcome], fontsize=FONT_SIZE, fontweight="bold")
    ax.legend(
        title="L1 Ratio",
        prop={"size": LEGEND_SIZE},
        title_fontsize=LEGEND_SIZE,
        loc="lower left"
    )
    ax.tick_params(axis="both", labelsize=FONT_SIZE)

plt.tight_layout()
plt.show()


## 1.3 Fit model with selected best parameters

In [ ]:
# Define test predictors
test_predictors = scaled_test_last.drop(columns=drop_vars)

# Evaluate each outcome
for outcome in ['amyloid', 'gpath', 'tangles', 'niareagansc']:
    print(f"Evaluating {outcome} on test data")
    
    # Define the target outcome
    train_outcome = scaled_train_last[outcome]
    test_outcome = scaled_test_last[outcome]

    # Retrieve best parameters
    best_params = grid_search_results[outcome]["best_params"]

    # Rebuild the model with best parameters
    best_model = utils.ElasticNetRidgeSwitcher(
        alpha=best_params['alpha'],
        l1_ratio=best_params['l1_ratio'],
        max_iter=10000
    )

    # Fit the model on the training data
    best_model.fit(train_predictors, train_outcome)

    # Predict on test data
    y_pred = best_model.predict(test_predictors)

    # Evaluate performance
    r2 = r2_score(test_outcome, y_pred)
    print(f"parameters: alpha {best_params['alpha']}, l1_ratio {best_params['l1_ratio']}")
    print(f"R-squared on test data for {outcome}: {r2:.4f}")
    print("-" * 50)